## Example 1: Basic tile operations with Web Mercator

In [ ]:
# Example 1: Basic tile operations with Web Mercator
import morecantile

print("Example 1 - Basic tile operations")

# Get Web Mercator tile matrix set
tms = morecantile.tms.get("WebMercatorQuad")
print(f"TMS: {tms.id}")

# Convert coordinates to tile (San Francisco)
lon, lat, zoom = -122.4194, 37.7749, 12
tile = tms.tile(lon, lat, zoom)
print(f"Coord ({lon}, {lat}) at zoom {zoom}")
print(f"Tile: ({tile.x}, {tile.y}, {tile.z})")

# Get tile bounds
bounds = tms.bounds(tile)
print(f"Bounds: ({bounds.left:.4f}, {bounds.bottom:.4f}, {bounds.right:.4f}, {bounds.top:.4f})")

# Calculate tile center
center_lon = (bounds.left + bounds.right) / 2
center_lat = (bounds.bottom + bounds.top) / 2
print(f"Center: ({center_lon:.4f}, {center_lat:.4f})")
assert tms.id == "WebMercatorQuad"
assert (tile.x, tile.y, tile.z) == (655, 1583, zoom), tile
assert bounds.left <= lon <= bounds.right and bounds.bottom <= lat <= bounds.top, bounds

print("Example 1 completed")

## Example 2: Multiple tile matrix sets

In [ ]:
# Example 2: Multiple tile matrix sets
import morecantile

print("Example 2 - Multiple tile matrix sets")

# List available TMS
available = morecantile.tms.list()
print(f"Available TMS: {len(available)}")
print(f"First 5: {available[:5]}")
assert {"WebMercatorQuad", "WorldCRS84Quad"} <= set(available)

# Compare projections for NYC
lon, lat, zoom = -74.0060, 40.7128, 10

for tms_name in ["WebMercatorQuad", "WorldCRS84Quad"]:
    tms = morecantile.tms.get(tms_name)
    tile = tms.tile(lon, lat, zoom)
    bounds = tms.bounds(tile)
    
    print(f"\n{tms_name}:")
    print(f"  Tile: ({tile.x}, {tile.y}, {tile.z})")
    print(f"  Bounds: ({bounds.left:.4f}, {bounds.bottom:.4f})")
    assert tile.z == zoom
    assert bounds.left <= lon <= bounds.right and bounds.bottom <= lat <= bounds.top, (tms_name, bounds)

print("Example 2 completed")

## Example 3: Tiles in bounding box

In [ ]:
# Example 3: Tiles in bounding box
import morecantile
from morecantile import BoundingBox

print("Example 3 - Tiles in bounding box")

tms = morecantile.tms.get("WebMercatorQuad")

# Define SF Bay Area bounding box
bbox = BoundingBox(-122.75, 37.45, -122.05, 37.85)
print(f"Bbox: {bbox}")

# Get tiles at different zooms
counts = []
for zoom in [8, 10, 12]:
    tiles = list(tms.tiles(bbox.left, bbox.bottom, bbox.right, bbox.top, zooms=[zoom]))
    print(f"\nZoom {zoom}: {len(tiles)} tiles")
    assert tiles and all(t.z == zoom for t in tiles)
    counts.append(len(tiles))
    
    # Show first tile
    if tiles:
        tile = tiles[0]
        print(f"  First: ({tile.x}, {tile.y}, {tile.z})")

# A higher zoom needs more tiles to cover the same box
assert counts == sorted(counts) and counts[0] < counts[-1], counts

print("Example 3 completed")

## Example 4: Tile parent and children

In [ ]:
# Example 4: Tile parent and children
import morecantile

print("Example 4 - Tile hierarchy")

tms = morecantile.tms.get("WebMercatorQuad")

# Base tile
base = tms.tile(-122.4194, 37.7749, 10)
print(f"Base (z=10): ({base.x}, {base.y}, {base.z})")

# Parent tile (zoom - 1)
parent_x = base.x // 2
parent_y = base.y // 2
parent = morecantile.Tile(parent_x, parent_y, base.z - 1)
print(f"Parent (z=9): ({parent.x}, {parent.y}, {parent.z})")

# Children tiles (zoom + 1)
children = [
    morecantile.Tile(base.x * 2, base.y * 2, base.z + 1),
    morecantile.Tile(base.x * 2 + 1, base.y * 2, base.z + 1),
    morecantile.Tile(base.x * 2, base.y * 2 + 1, base.z + 1),
    morecantile.Tile(base.x * 2 + 1, base.y * 2 + 1, base.z + 1)
]

print("Children (z=11):")
for i, child in enumerate(children, 1):
    print(f"  {i}. ({child.x}, {child.y}, {child.z})")

# The hand-computed tiles match morecantile's own parent() and children()
assert tms.parent(base) == [parent], tms.parent(base)
assert sorted(tms.children(base)) == sorted(children), tms.children(base)

print("Example 4 completed")

## Example 5: Tile URL patterns and quadkeys

In [ ]:
# Example 5: Tile URL patterns and quadkeys
import morecantile

print("Example 5 - Tile URLs and quadkeys")

tms = morecantile.tms.get("WebMercatorQuad")

# Get tile for San Francisco
tile = tms.tile(-122.4194, 37.7749, 12)
print(f"Tile: ({tile.x}, {tile.y}, {tile.z})")

# Generate tile URLs
url_template = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"
url = url_template.format(x=tile.x, y=tile.y, z=tile.z)
print(f"OSM URL: {url}")

# Calculate quadkey (Bing tiles format)
quadkey = ""
for i in range(tile.z, 0, -1):
    digit = 0
    mask = 1 << (i - 1)
    if (tile.x & mask) != 0:
        digit += 1
    if (tile.y & mask) != 0:
        digit += 2
    quadkey += str(digit)
print(f"Quadkey: {quadkey}")
assert url == "https://tile.openstreetmap.org/12/655/1583.png", url
# The hand-computed quadkey matches morecantile's, and decodes back to the same tile
assert len(quadkey) == tile.z and quadkey == tms.quadkey(tile)
assert tms.quadkey_to_tile(quadkey) == tile

print("Example 5 completed")